# SmartCito Training Demo

This notebook is the Kaggle training walkthrough for **SmartCito Model**. It shows how contributors can prepare SmartCito data, configure a compatible base model from an official provider source, run LoRA or QLoRA fine-tuning, evaluate the adapter, and export the artifacts needed for collaboration.

This bundle does not include LLaMA-3 weights. It only ships SmartCito code, LoRA or QLoRA adapters, and synthetic or sovereign datasets. Users must obtain any compatible base model from official provider sources.

What contributors can do:
- Prepare SmartCito records from synthetic samples or sovereign edge logs
- Train SmartCito adapters on drone, robot, camera, sensor, threat, and geographic tasks
- Evaluate adapter behavior on operational SmartCito domains
- Export adapter-only artifacts for safe sharing

In [ ]:
from pathlib import Path
import json
import os
import subprocess

ROOT = Path.cwd()
DATASET_PATH = ROOT / 'datasets' / 'sample_training_data.json'
PREPARED_PATH = ROOT / 'datasets' / 'prepared_smartcito_training_data.jsonl'
ADAPTER_PATH = ROOT / 'output' / 'smartcito-lora'
BASE_MODEL_ID = os.getenv('SMARTCITO_BASE_MODEL_ID', '').strip()
RUN_TRAINING = False
RUN_EVALUATION = False

print(json.dumps({
    'base_model_id': BASE_MODEL_ID or 'set SMARTCITO_BASE_MODEL_ID',
    'dataset_path': str(DATASET_PATH),
    'prepared_dataset_path': str(PREPARED_PATH),
    'adapter_path': str(ADAPTER_PATH),
    'run_training': RUN_TRAINING,
    'run_evaluation': RUN_EVALUATION,
}, indent=2))

## 1. Setup

Install the SmartCito dependencies, set your Hugging Face token if the base model is gated, and confirm the working paths before training. In Kaggle, place the SmartCito dataset bundle under `/kaggle/input/...` and update `ROOT` if needed.

In [ ]:
install_command = ['python3', '-m', 'pip', 'install', '-r', str(ROOT / 'ai_models' / 'requirements.txt')]
print('Install dependencies with:')
print(' '.join(install_command))

if os.getenv('HF_TOKEN') or os.getenv('HUGGINGFACE_HUB_TOKEN'):
    print('Hugging Face token detected in environment.')
else:
    print('Set HF_TOKEN or HUGGINGFACE_HUB_TOKEN before loading a gated base model.')

## 2. Load SmartCito Dataset

The sample training set includes operational tasks across drone logs, robot navigation, camera anomaly samples, environmental sensor fusion, and geographic deployment reasoning.

In [ ]:
records = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
domains = sorted({record.get('metadata', {}).get('domain', 'unknown') for record in records})
preview = [{
    'instruction': record['instruction'],
    'domain': record.get('metadata', {}).get('domain', 'unknown')
} for record in records[:5]]

print(json.dumps({
    'records': len(records),
    'domains': domains,
    'preview': preview,
}, indent=2))

## 3. Fine-Tuning with LoRA or QLoRA

Prepare the chat-formatted dataset, then run either LoRA or QLoRA. The commands below use small example settings for a demonstration pass. Set `RUN_TRAINING = True` only when the notebook environment has GPU capacity and the required base model access.

In [ ]:
prepare_command = [
    'python3', 'ai/training/prepare_dataset.py',
    '--input', str(DATASET_PATH),
    '--output', str(PREPARED_PATH),
]
train_command = [
    'python3', 'ai/training/lora_training.py',
    '--dataset', str(DATASET_PATH),
    '--output-dir', str(ADAPTER_PATH),
    '--epochs', '0.1',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '2',
]
qlora_command = [
    'python3', 'ai/training/qlora_training.py',
    '--dataset', str(DATASET_PATH),
    '--output-dir', str(ADAPTER_PATH),
    '--epochs', '0.1',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '2',
]

if BASE_MODEL_ID:
    train_command.extend(['--base-model', BASE_MODEL_ID])
    qlora_command.extend(['--base-model', BASE_MODEL_ID])

print('Prepare command:')
print(' '.join(prepare_command))
print('LoRA command:')
print(' '.join(train_command) if BASE_MODEL_ID else 'Set SMARTCITO_BASE_MODEL_ID before training.')
print('QLoRA command:')
print(' '.join(qlora_command) if BASE_MODEL_ID else 'Set SMARTCITO_BASE_MODEL_ID before QLoRA training.')

subprocess.run(prepare_command, check=True)
if RUN_TRAINING and BASE_MODEL_ID:
    subprocess.run(train_command, check=True)

## 4. Evaluation

Evaluate the trained adapter on SmartCito tasks such as drone mission planning, sensor fusion, threat detection, city camera analytics, and geographic reasoning.

In [ ]:
evaluate_command = [
    'python3', 'ai/training/evaluate_adapters.py',
    '--dataset', 'ai/datasets/sample_evaluation_data.json',
    '--adapter-path', str(ADAPTER_PATH),
    '--output', str(ADAPTER_PATH / 'evaluation_summary.json'),
    '--markdown-report', str(ADAPTER_PATH / 'evaluation_report.md'),
]

if BASE_MODEL_ID:
    evaluate_command.extend(['--base-model', BASE_MODEL_ID])

print('Evaluation command:')
print(' '.join(evaluate_command) if BASE_MODEL_ID else 'Set SMARTCITO_BASE_MODEL_ID before live evaluation, or score from a predictions file.')

if RUN_EVALUATION and ADAPTER_PATH.exists() and BASE_MODEL_ID:
    subprocess.run(evaluate_command, check=True)
else:
    print('Set RUN_EVALUATION = True and SMARTCITO_BASE_MODEL_ID after training to execute live evaluation.')

## 5. Export

Share only the adapter directory and evaluation outputs. Do **not** upload foundation-model weights, proprietary data, or secrets. The Kaggle notebook deliverables in this bundle are:

- `ai/examples/SmartCito_Training_Demo.ipynb`
- `ai/examples/smartcito_inference_demo.ipynb`

Optional export artifacts:
- `ai/output/smartcito-lora/`
- `ai/output/smartcito-lora/training_summary.json`
- `ai/output/smartcito-lora/evaluation_summary.json`
- `ai/output/smartcito-lora/evaluation_report.md`